# Session 1: the way you sort cards is an algorithm
**A. Real-world problem.** You hold a hand of playing cards and put them in order. You do
not think about it, yet you follow rules. Write the rules down exactly and a computer can
follow them on a million cards. Two simple rule sets are *bubble sort* and *insertion sort*.

**Plan.** (1) write the two rule sets as Python, (2) count the steps they take
(comparisons and swaps), (3) see that the steps depend on how mixed up the cards start,
(4) draw the cards as bars before and after.

In [ ]:
import pathlib

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: bubble sort
Look at two neighbouring cards. If the left one is bigger, swap them. Walk along the row
doing this; the biggest card "bubbles" to the right end. Repeat on the shorter row that is left.
Stop early if a whole walk needed no swap, because then the row is already in order.

We count every *comparison* (looking at two cards) and every *swap* (exchanging two neighbours).

In [ ]:
def bubble_sort(values):
    """Return (sorted list, comparisons, swaps). The input list is not changed."""
    a = list(values)
    comparisons = swaps = 0
    for end in range(len(a) - 1, 0, -1):
        swapped = False
        for j in range(end):
            comparisons += 1
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                swaps += 1
                swapped = True
        if not swapped:
            break
    return a, comparisons, swaps

## Step 2: insertion sort
This is how most people sort a hand of cards. Take the next card and slide it left past every
bigger card until it fits. The cards on the left are always in order.
Each one-place slide is counted as a swap (a one-place move).

In [ ]:
def insertion_sort(values):
    """Return (sorted list, comparisons, swaps). The input list is not changed."""
    a = list(values)
    comparisons = swaps = 0
    for i in range(1, len(a)):
        j = i
        while j > 0:
            comparisons += 1
            if a[j - 1] > a[j]:
                a[j - 1], a[j] = a[j], a[j - 1]
                swaps += 1
                j -= 1
            else:
                break
    return a, comparisons, swaps

## Step 3: how mixed up is the row?
An *inversion* is a pair of cards that is in the wrong order (the bigger one is to the left).
Each swap of two neighbours removes exactly one inversion, so the number of swaps both methods
need equals the number of inversions. A sorted row has 0 inversions; a reversed row of n cards
has the most, n(n-1)/2.

In [ ]:
def inversions(values):
    a = list(values)
    return sum(1 for i in range(len(a)) for j in range(i + 1, len(a)) if a[i] > a[j])


def prefix_reversed(n, k):
    """Cards 1..n in order, except that the first k cards are reversed (k = 0 or 1 changes nothing)."""
    k = max(0, min(k, n))
    return list(range(k, 0, -1)) + list(range(k + 1, n + 1))


def is_sorted(values):
    return all(values[i] <= values[i + 1] for i in range(len(values) - 1))

## Step 4: see it
Left: a row of 12 cards drawn as bars, before and after. Right: the steps each method takes as
the first k cards of a 30-card row are reversed (k = 0 is already sorted, k = 30 is fully reversed).

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    hand = [7, 3, 11, 1, 9, 5, 12, 2, 10, 4, 8, 6]
    result, c, s = bubble_sort(hand)
    print("12 cards:", hand, "->", result, "| comparisons", c, "swaps", s, "inversions", inversions(hand))
    fig, axs = plt.subplots(1, 2, figsize=(8, 3.2), sharey=True)
    axs[0].bar(range(len(hand)), hand, color="#d9534f")
    axs[0].set_title("before: %d inversions" % inversions(hand))
    axs[1].bar(range(len(result)), result, color="#2a9d8f")
    axs[1].set_title("after: 0 inversions")
    for ax in axs:
        ax.set_xlabel("position in the row")
    axs[0].set_ylabel("card value")
    fig.tight_layout()
    fig.savefig(OUT / "bars_before_after.png", dpi=120)

    n = 30
    ks = list(range(0, n + 1))
    bc, bs, ic, is_ = [], [], [], []
    for k in ks:
        row = prefix_reversed(n, k)
        bc.append(bubble_sort(row)[1]); bs.append(bubble_sort(row)[2])
        ic.append(insertion_sort(row)[1]); is_.append(insertion_sort(row)[2])
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(ks, bc, label="bubble comparisons")
    ax2.plot(ks, ic, label="insertion comparisons")
    ax2.plot(ks, bs, "--", label="swaps (both methods)")
    ax2.set_xlabel("k = how many leading cards are reversed (30 cards)")
    ax2.set_ylabel("steps counted")
    ax2.set_title("Steps grow with the disorder")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "steps_vs_disorder.png", dpi=120)
    print("fully reversed 30 cards: bubble", bubble_sort(prefix_reversed(30, 30))[1:], "insertion", insertion_sort(prefix_reversed(30, 30))[1:])
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Take the cards 3 1 2. Predict the comparisons and swaps of bubble sort and of insertion sort, then run both functions in `lesson.py` to check.
2. **Stretch.** Change the 12-card hand in `lesson.py` to a hand of your own and use `inversions` to predict the number of swaps before you run the sorts.
3. **Explain.** Explain, with the cards 2 1 3 4 5 and 1 2 3 5 4, why bubble sort can do many more comparisons than insertion sort on a row that is almost sorted.

## Assessment
- **A. Conceptual.** Why do bubble sort and insertion sort need exactly the same number of swaps for the cards 5 2 4 1 3, even though they work differently?
- **B. Numerical.** A row of 30 cards is completely reversed (30 down to 1). How many comparisons and swaps does insertion sort need? Check your answer with a formula.
- **C. Graph and simulation interpretation.** In the steps graph for 30 cards, at k = 10 (the first 10 cards reversed), which method needs fewer comparisons and by how many? How many swaps do they need?
- **D. Design and optimisation challenge.** Design: a teacher hands you 30 cards that are sorted except that the first 20 are in reverse order. You must pick bubble sort or insertion sort and tell the class how many comparisons to expect. Which do you pick and why? What would you pick if the cards were already sorted?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    cards = [5, 2, 4, 1, 3]
    b = L.bubble_sort(cards)
    i = L.insertion_sort(cards)
    checks.append(("worked example: bubble sort of 5 2 4 1 3 gives 1 2 3 4 5 with 10 comparisons and 7 swaps",
                   b == ([1, 2, 3, 4, 5], 10, 7)))
    checks.append(("worked example: insertion sort of 5 2 4 1 3 takes 9 comparisons and 7 swaps",
                   i == ([1, 2, 3, 4, 5], 9, 7)))
    checks.append(("5 2 4 1 3 has 7 pairs in the wrong order, equal to the swaps", L.inversions(cards) == 7))
    checks.append(("the input list is not changed", cards == [5, 2, 4, 1, 3]))

    rev = list(range(30, 0, -1))
    checks.append(("reversed 30 cards: 435 = 30*29/2 comparisons and swaps for both methods",
                   L.bubble_sort(rev)[1:] == (435, 435) and L.insertion_sort(rev)[1:] == (435, 435)))
    srt = list(range(1, 31))
    checks.append(("sorted 30 cards: 29 comparisons and 0 swaps for both methods",
                   L.bubble_sort(srt)[1:] == (29, 0) and L.insertion_sort(srt)[1:] == (29, 0)))
    checks.append(("empty list and one card need 0 steps",
                   L.bubble_sort([]) == ([], 0, 0) and L.insertion_sort([7]) == ([7], 0, 0)))

    row = L.prefix_reversed(30, 10)
    checks.append(("30 cards with the first 10 reversed: 45 inversions, bubble 245 comparisons, insertion 65",
                   L.inversions(row) == 45 and L.bubble_sort(row)[1] == 245 and L.insertion_sort(row)[1] == 65))
    row20 = L.prefix_reversed(30, 20)
    checks.append(("first 20 reversed: 190 swaps and 20*19/2 = 190 inversions",
                   L.bubble_sort(row20)[2] == 190 == L.inversions(row20) == 20 * 19 // 2))
    checks.append(("first 20 of 30 reversed: bubble 390 comparisons, insertion 200",
                   L.bubble_sort(row20)[1] == 390 and L.insertion_sort(row20)[1] == 200))
    hand = [7, 3, 11, 1, 9, 5, 12, 2, 10, 4, 8, 6]
    checks.append(("12-card hand: 32 inversions, bubble 56 comparisons, insertion 41 comparisons, both 32 swaps",
                   L.inversions(hand) == 32 and L.bubble_sort(hand) == (sorted(hand), 56, 32)
                   and L.insertion_sort(hand) == (sorted(hand), 41, 32)))
    checks.append(("prefix_reversed(5, 3) is 3 2 1 4 5", L.prefix_reversed(5, 3) == [3, 2, 1, 4, 5]))
    checks.append(("prefix_reversed with k = 0 or 1 changes nothing",
                   L.prefix_reversed(6, 0) == L.prefix_reversed(6, 1) == [1, 2, 3, 4, 5, 6]))
    checks.append(("is_sorted works", L.is_sorted([1, 2, 2, 3]) and not L.is_sorted([2, 1])))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
